# Generic VAD Adapter

> The generic (tool-agnostic) VAD adapter — cache-check, invoke the bound tool's pure-compute `detect_speech`, persist. Reused across every tool capability satisfying `VADToolProtocol`, exactly as `GenericTranscriptionAdapter` is reused across transcribers.

**Native-surface model (stage 8 / PILLAR 1c):** the TOOL is pure compute; this adapter adds the cache + persistence bookends + the per-call `force` control. Mirrors `GenericTranscriptionAdapter`; the only shape difference is the content-addressed cache row carries no `job_id` (VAD never did — see `VADStorage`).

In [ ]:
#| default_exp generic

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
from pathlib import Path
from typing import Union

from cjm_plugin_system.core.wire import get_call_envelope
from cjm_plugin_system.utils.hashing import hash_file, hash_dict_canonical

from cjm_capability_primitives.vad import VADResult, TimeRange
from cjm_vad_adapter_interface.adapter import VADAdapter
from cjm_vad_adapter_interface.storage import VADStorage

In [ ]:
#| export
class GenericVADAdapter(VADAdapter):
    """Generic VAD adapter: cache-check -> pure-compute tool -> persist.

    Works against ANY tool satisfying `VADToolProtocol`. The bookends:

      1. cache check (file_path + file_hash + config_hash) BEFORE invoking the
         tool, so a hit never loads the model;
      2. the tool's pure-compute `detect_speech` on a miss / forced call;
      3. `save_with_logging` (upsert by file_path + config_hash).

    `config_hash` reuses `hash_dict_canonical(get_current_config())` (the SAME
    canonical hash the fused-era plugin used). `force` rides
    `CallEnvelope.control` (not a task kwarg, keeping `detect_speech(audio)`
    pure). Storage lives at `<PLUGIN_DATA_DIR>/vad.db`; the substrate injects
    the per-capability `PLUGIN_DATA_DIR` at spawn, so the adapter neither
    hard-codes a path nor asks the tool for one.
    """

    def _get_storage(self) -> VADStorage:
        """Lazily open the per-capability cache DB under the substrate-injected
        PLUGIN_DATA_DIR (created at worker spawn)."""
        if getattr(self, "_storage", None) is None:
            db_path = os.path.join(os.environ["PLUGIN_DATA_DIR"], "vad.db")
            self._storage = VADStorage(db_path)
        return self._storage

    def detect_speech(
        self,
        audio: Union[str, Path],  # Path to MODEL-READY audio (converted upstream)
        **kwargs,                 # Provenance + tool options
    ) -> VADResult:               # Typed VAD output
        """Cache-check, invoke the bound tool's pure-compute `detect_speech`, persist."""
        file_path = str(audio)
        file_hash = hash_file(file_path)
        config_hash = hash_dict_canonical(self.tool.get_current_config())

        env = get_call_envelope()
        force = bool(env.control.get("force")) if env is not None else False

        storage = self._get_storage()
        if not force:
            cached = storage.get_cached(file_path, file_hash, config_hash)
            if cached is not None:
                return VADResult(
                    ranges=[TimeRange(**r) for r in (cached.ranges or [])],
                    metadata=cached.metadata or {},
                )

        # Pure compute on the tool (cache miss / forced).
        result = self.tool.detect_speech(audio, **kwargs)

        # Persist. Content-addressed row (no job_id); cache key is
        # (file_path, config_hash) with file_hash for content-correctness.
        storage.save_with_logging(
            file_path=file_path, file_hash=file_hash, config_hash=config_hash,
            ranges=[r.to_dict() for r in result.ranges], metadata=result.metadata,
        )
        return result

In [ ]:
# Cache-bookends test: a fake PURE-COMPUTE tool; miss -> compute + persist,
# hit -> NO recompute (and ranges re-type to TimeRange), force (via
# CallEnvelope.control) -> bypass + recompute.
import os, tempfile
from cjm_plugin_system.core.wire import CallEnvelope, set_call_envelope, reset_call_envelope

os.environ["PLUGIN_DATA_DIR"] = tempfile.mkdtemp(prefix="cjm_gva_test_")

class _FakeTool:
    def __init__(self): self.calls = 0
    def get_current_config(self): return {"threshold": 0.5}
    def detect_speech(self, audio, **kwargs) -> VADResult:
        self.calls += 1
        return VADResult(ranges=[TimeRange(start=0.0, end=float(self.calls))],
                         metadata={"calls": self.calls})

_f = tempfile.NamedTemporaryFile(suffix=".wav", delete=False); _f.write(b"audiobytes"); _f.close()

tool = _FakeTool()
ad = GenericVADAdapter(tool)

# 1. cache MISS -> compute (calls=1) + persist
r1 = ad.detect_speech(_f.name)
assert tool.calls == 1 and r1.ranges[0].end == 1.0

# 2. cache HIT -> no recompute; ranges re-typed to TimeRange (NOT dicts)
r2 = ad.detect_speech(_f.name)
assert tool.calls == 1, f"expected cache hit, calls={tool.calls}"
assert isinstance(r2.ranges[0], TimeRange) and r2.ranges[0].end == 1.0

# 3. force via CallEnvelope.control -> bypass cache, recompute (calls=2)
_tok = set_call_envelope(CallEnvelope(job_id="j3", control={"force": True}))
try:
    r3 = ad.detect_speech(_f.name)
finally:
    reset_call_envelope(_tok)
assert tool.calls == 2, f"force should bypass cache, calls={tool.calls}"
print("GenericVADAdapter cache/force bookends OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()